In [ ]:
import re
import unicodedata
import pandas as pd
import io
import ipywidgets as widgets
from IPython.display import display, clear_output

def normalizar_caracteres_especiales(texto):
    if not isinstance(texto, str):
        return ""
    texto_protegido = texto.replace('Ñ', '[[MAY_N_TILDE]]').replace('ñ', '[[MIN_N_TILDE]]')
    texto_protegido = re.sub(r'[Ââ]', lambda m: 'A' if m.group().isupper() else 'a', texto_protegido)
    nfkd_form = unicodedata.normalize('NFKD', texto_protegido)
    texto_sin_diacriticos = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    return texto_sin_diacriticos.replace('[[MAY_N_TILDE]]', 'Ñ').replace('[[MIN_N_TILDE]]', 'ñ')

def normalizar_direccion(direccion):
    if pd.isna(direccion) or not str(direccion).strip():
        return ""
    texto_espacios = re.sub(r'\s+', ' ', str(direccion), flags=re.UNICODE)
    texto_normalizado = normalizar_caracteres_especiales(texto_espacios)
    texto_filtrado = re.compile(r'[^A-Za-zÑñ0-9 #\-]').sub('', texto_normalizado)
    return re.sub(r' {2,}', ' ', texto_filtrado).strip()

# Componentes de la interfaz gráfica
uploader = widgets.FileUpload(accept='.xlsx', multiple=False, description="Buscar Excel")
btn_procesar = widgets.Button(description="Procesar y Descargar", button_style='success', icon='gear')
output_area = widgets.Output()

def ejecutar_normalizacion(b):
    with output_area:
        clear_output()
        if not uploader.value:
            print("❌ Por favor, primero selecciona un archivo Excel (.xlsx).")
            return
        
        print("⏳ Leyendo y procesando el archivo, por favor espera...")
        
        try:
            # Corrección del error de extracción para ipywidgets moderno
            uploaded_file = uploader.value[0]
            nombre_archivo_original = uploaded_file['name']
            contenido_archivo = uploaded_file['content']
            
            # Convertir memoryview a bytes
            contenido_bytes = bytes(contenido_archivo)
            
            df = pd.read_excel(io.BytesIO(contenido_bytes), dtype=str, engine='openpyxl')
        except Exception as e:
            print(f"❌ Error al abrir el Excel: {e}")
            return

        columna_objetivo = 'Direccion'
        if columna_objetivo not in df.columns:
            print(f"❌ [ERROR] No se encontró la columna exacta '{columna_objetivo}'.")
            print(f"Columnas en tu archivo: {list(df.columns)}")
            return

        df['Direccion ajustada'] = df[columna_objetivo].apply(normalizar_direccion)
        
        # Preparar archivo de salida en memoria para descarga inmediata
        output_stream = io.BytesIO()
        with pd.ExcelWriter(output_stream, engine='openpyxl') as writer:
            df.to_excel(writer, index=False)
        output_stream.seek(0)
        
        # Crear botón de descarga web nativo para el navegador del usuario
        nombre_salida = nombre_archivo_original.replace(".xlsx", "_ajustado.xlsx")
        downloader = widgets.DownloadButton(
            filename=nombre_salida,
            value=output_stream.read(),
            description="Descargar Resultado",
            button_style='info',
            icon='download'
        )
        
        clear_output()
        print("✅ ¡Proceso completado exitosamente!\n")
        print(f"Filas totales procesadas: {len(df)}")
        print("Haz clic abajo para guardar tu archivo:")
        display(downloader)

btn_procesar.on_click(ejecutar_normalizacion)

# Maquetación visual
titulo = widgets.HTML("<h2>Normalizador de Direcciones - Luker</h2>")
instrucciones = widgets.HTML("<p>1. Selecciona tu archivo Excel.<br>2. Presiona 'Procesar y Descargar' para limpiar las direcciones.</p>")
interfaz = widgets.VBox([titulo, instrucciones, uploader, widgets.HTML("<br>"), btn_procesar, widgets.HTML("<br>"), output_area])
display(interfaz)
